# Asistente de normativa de tráfico y permisos de moto (RAG)

Proyecto final de IA Generativa: un asistente experto en la normativa de tráfico española y, en concreto, en los permisos de moto, construido con **RAG** (*Retrieval-Augmented Generation*).

**Stack:** Google Gemini (LLM y embeddings) · ChromaDB · LangGraph + LangChain · Jupyter · Streamlit.

---
# 1. Base de conocimiento

En esta sección se construye la base de conocimiento: carga de los PDF del BOE, limpieza, troceado (*chunking*), embeddings e indexación en ChromaDB. Al final se comprueba que la recuperación funciona **sin usar todavía el LLM**.

Todo el código está en `src/knowledge_base.py`; el notebook lo usa y explica cada decisión.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import knowledge_base as kb

pd.set_option("display.max_colwidth", 120)
print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /home/josele/Documentos/projects/proyecto_ia_generativa


## 1.1 Fuentes

Se usan los **textos consolidados** del BOE (incluyen todas las modificaciones posteriores, así que son la versión vigente):

| Norma | Referencia | Contenido relevante |
|---|---|---|
| Reglamento General de Conductores | RD 818/2009 (BOE-A-2009-9481) | Clases de permiso (AM, A1, A2, A, B…), edades mínimas, exámenes |
| Reglamento General de Circulación | RD 1428/2003 (BOE-A-2003-23514) | Normas de circulación: velocidades, casco, adelantamientos… |
| Ley de Tráfico y Seguridad Vial | RDL 6/2015 (BOE-A-2015-11722) | Infracciones, sanciones y permiso por puntos |

Los PDF se han descargado directamente de boe.es a la carpeta `data/`. **La API key de Gemini se lee de `.env` (`GOOGLE_API_KEY`) y nunca aparece en el código.**

In [2]:
for f, info in kb.DOCUMENTS.items():
    p = kb.DATA_DIR / f
    print(f"{info['norma']:<38} {f:<36} {p.stat().st_size / 1e6:7.1f} MB")

Reglamento General de Conductores      BOE-A-2009-9481-consolidado.pdf          3.8 MB
Reglamento General de Circulación      BOE-A-2003-23514-consolidado.pdf       258.7 MB
Ley de Tráfico y Seguridad Vial        BOE-A-2015-11722-consolidado.pdf         0.6 MB


## 1.2 Carga y limpieza

Cada PDF se carga con `PyPDFLoader` (un `Document` por página). El texto que extrae `pypdf` del BOE trae ruido que conviene quitar antes de trocear:

- **Cabeceras y pies repetidos**: cada página termina con `BOLETÍN OFICIAL DEL ESTADO` / `LEGISLACIÓN CONSOLIDADA` / `Página N`. Si no se quitan, aparecen en casi todos los chunks, no aportan significado y «contaminan» los embeddings (todos los chunks se parecen un poco entre sí por culpa de ese texto).
- **Páginas de índice**: las primeras páginas son el índice, con líneas del tipo `Artículo 4. Edad mínima ........ 12`. Se descartan porque mencionan todos los artículos sin su contenido y se colarían en los resultados del retrieval. Se detectan porque tienen muchas líneas con puntos guía.
- **Páginas sin texto**: el Reglamento de Circulación tiene unas 600 páginas que son **imágenes** (el catálogo de señales). No tienen texto extraíble y se descartan. *Limitación conocida*: el asistente no «ve» las señales gráficas, solo su descripción en el articulado.
- **Palabras cortadas con guion** al final de línea (`conduc-⏎ción` → `conducción`).
- **Tabla de clases de permiso** (art. 4 del Reglamento de Conductores): `pypdf` extrae las tablas como líneas sueltas: `A2` / *Motocicletas con una potencia máxima de 35 kW…* / `18`. En la primera versión, el chunk que contenía «A2 … 18» empezaba a mitad de la tabla, sin el título del artículo, y **no aparecía en el top-5** para «edad mínima A2» (ver el análisis del final). Como esta tabla es el núcleo del proyecto (clases AM, A1, A2, A y B), se reescribe cada fila como texto autocontenido: «*Permiso de la clase A2. Vehículos que autoriza a conducir: Motocicletas… Edad mínima para el permiso de la clase A2: 18 años cumplidos.*». Se hace solo en esa tabla y sin inventar texto: solo se etiquetan las columnas (clase, vehículos, edad).
- **Espacios**: espacios duros (NBSP), guiones blandos y espacios múltiples se normalizan. Además, los saltos de línea a mitad de frase (la línea siguiente empieza en minúscula) se convierten en espacios, para que cada chunk sea prosa continua. Los saltos que preceden a «Artículo», apartados (`1.`, `a)`) o títulos se mantienen, porque los usa el *splitter*.

Las páginas limpias de cada documento se **concatenan** (guardando dónde empieza cada página). Así un artículo que salta de una página a otra no se corta artificialmente y se puede saber la página de cada chunk.

Ejemplo de una página antes y después de la limpieza:

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader(str(kb.DATA_DIR / "BOE-A-2009-9481-consolidado.pdf")).load()
raw = pages[15].page_content
print("=== ORIGINAL (final de la página 16) ===")
print(raw[-600:])
print("\n=== LIMPIO ===")
print(kb.clean_text(raw)[-450:])

=== ORIGINAL (final de la página 16) ===
 
grupo 2, según la clasificación establecida en el artículo 45, implicará la de las 
autorizaciones del grupo 1 de las que sea titular el interesado, por los plazos que a éstas 
correspondan.
La solicitud de prórroga de vigencia del permiso o licencia podrá presentarse con una 
antelación máxima de tres meses a su fecha de caducidad, computándose desde esta última 
fecha el nuevo período de vigencia de la autorización. En supuestos excepcionales 
debidamente justificados, se podrá solicitar con una antelación mayor, computándose el 
BOLETÍN OFICIAL DEL ESTADO
LEGISLACIÓN CONSOLIDADA
Página 16

=== LIMPIO ===
s autorizaciones del grupo 1 de las que sea titular el interesado, por los plazos que a éstas correspondan.
La solicitud de prórroga de vigencia del permiso o licencia podrá presentarse con una antelación máxima de tres meses a su fecha de caducidad, computándose desde esta última fecha el nuevo período de vigencia de la autorización. En su

In [4]:
docs = kb.load_documents()

stats_pages = pd.DataFrame([{
    "Norma": d.norma,
    "Páginas PDF": d.total_pages,
    "Páginas con texto útil": d.text_pages,
    "Caracteres (limpio)": len(d.text),
} for d in docs])
stats_pages

,Norma,Páginas PDF,Páginas con texto útil,Caracteres (limpio)
0,Reglamento General de Conductores,109,97,352144
1,Reglamento General de Circulación,730,109,338376
2,Ley de Tráfico y Seguridad Vial,74,66,237027


## 1.3 Chunking

Se usa `RecursiveCharacterTextSplitter` con:

- **`chunk_size = 1000` caracteres** (~250 tokens). Un artículo corto o un apartado de uno largo cabe entero: es lo bastante grande para conservar el contexto de una norma (p. ej. «el permiso A2 autoriza… *y la edad mínima es…*») y lo bastante pequeño para que el embedding represente una sola idea y no se diluya mezclando varios temas. Con k=5 el contexto que llegará al LLM (~5.000 caracteres) es manejable.
- **`chunk_overlap = 150` caracteres** (~15 %). Cuando un artículo largo se parte, el solapamiento evita que una frase quede cortada entre dos chunks sin que ninguno la tenga completa.
- **Separadores con prioridad jurídica**: el *splitter* prueba los separadores en orden y solo baja al siguiente si el trozo sigue siendo demasiado grande:
  1. inicio de `Artículo N.` (también `bis`/`ter` y `Artículo único.`)
  2. inicio de `Disposición adicional/transitoria/derogatoria/final`
  3. inicio de `ANEXO`
  4. `TÍTULO`, `CAPÍTULO`, `Sección`
  5. `Permiso de la clase …` (cada fila de la tabla de permisos)
  6. salto de línea (apartados `1.`, `a)`…), fin de frase, espacio y, como último recurso, carácter.

  Los separadores son expresiones regulares con *lookahead* y `keep_separator="start"`, de forma que el encabezado «Artículo N. Título» queda al **principio** del chunk que lo contiene y no al final del anterior. Los artículos cortos consecutivos se pueden juntar en un mismo chunk (hasta 1000 caracteres).

## 1.4 Metadatos

Cada chunk lleva estos metadatos, que permitirán **citar la fuente** en las respuestas y filtrar en el futuro:

| Campo | Ejemplo | Cómo se obtiene |
|---|---|---|
| `norma` | Reglamento General de Conductores | tabla fija por fichero |
| `referencia` | RD 818/2009 (BOE-A-2009-9481) | tabla fija por fichero |
| `fuente` | BOE-A-2009-9481-consolidado.pdf | nombre del fichero |
| `pagina` / `pagina_fin` | 13 / 14 | posición del chunk en el texto concatenado → página |
| `articulo` | Artículo 4 | último encabezado (`Artículo N.`, `Disposición…`, `ANEXO…`) que aparece **antes o al inicio** del chunk; si el chunk es la continuación de un artículo largo, hereda el número de ese artículo |
| `articulos` | Artículo 4, Artículo 5 | todos los artículos presentes en el chunk (si se han juntado varios) |
| `titulo` | Artículo 4. Clases de permiso de conducción y edad requerida para obtenerlo. | línea completa del encabezado vigente |
| `chunk_id` | BOE-A-2009-9481-consolidado-00027-a525fe8f02 | posición + hash del contenido (si cambia el chunking, cambia el ID) |

**Cabecera de contexto.** A cada chunk se le antepone `[norma · título del artículo]`, por ejemplo `[Reglamento General de Conductores · Artículo 4. Clases de permiso de conducción y edad requerida para obtenerlo.]`. Un chunk que es la continuación de un artículo largo no contiene el título del artículo, así que su embedding pierde el tema («edad», «permiso»…). La cabecera se lo devuelve y, además, le dirá al LLM de dónde sale cada fragmento (útil para citar).

Los encabezados solo se reconocen **a principio de línea** (las referencias en mitad del texto, como «según el artículo 77.h)», no cuentan). Los chunks anteriores al primer artículo se etiquetan como `Preámbulo`.

In [5]:
chunks = kb.build_chunks(docs)

stats_chunks = (
    pd.DataFrame([{"Norma": c.metadata["norma"], "len": len(c.page_content),
                   "con_articulo": c.metadata["articulo"] != "Preámbulo"} for c in chunks])
    .groupby("Norma", sort=False)
    .agg(Chunks=("len", "size"), **{"Longitud media": ("len", "mean")},
         **{"Longitud máx.": ("len", "max")}, **{"% con artículo": ("con_articulo", "mean")})
    .round({"Longitud media": 0, "% con artículo": 3})
)
stats_chunks["% con artículo"] *= 100
display(stats_pages.merge(stats_chunks, left_on="Norma", right_index=True))
print(f"Total de páginas PDF: {stats_pages['Páginas PDF'].sum()}  |  Total de chunks: {len(chunks)}")

,Norma,Páginas PDF,Páginas con texto útil,Caracteres (limpio),Chunks,Longitud media,Longitud máx.,% con artículo
0,Reglamento General de Conductores,109,97,352144,467,789.0,999,96.4
1,Reglamento General de Circulación,730,109,338376,509,680.0,999,98.4
2,Ley de Tráfico y Seguridad Vial,74,66,237027,345,709.0,998,98.0


Total de páginas PDF: 913  |  Total de chunks: 1321


In [6]:
# Ejemplo: el chunk con la fila del permiso A2 (tabla normalizada + cabecera de contexto)
ejemplo = next(c for c in chunks if "clase A2: 18 años" in c.page_content)
print(ejemplo.metadata)
print()
print(ejemplo.page_content)

{'norma': 'Reglamento General de Conductores', 'referencia': 'RD 818/2009 (BOE-A-2009-9481)', 'fuente': 'BOE-A-2009-9481-consolidado.pdf', 'pagina': 11, 'pagina_fin': 12, 'articulo': 'Artículo 4', 'articulos': 'Artículo 4', 'chunk_id': 'BOE-A-2009-9481-consolidado-00026'}

Artículo 4. Clases de permiso de conducción y edad requerida para obtenerlo.
1. Todas las clases de permiso de conducción de las que sea titular una persona deberán constar en un único documento con expresión de las categorías de vehículos cuya conducción autorizan.
2. El permiso de conducción será de las siguientes clases:
Clase de permiso Vehículos que autoriza a conducir Edad: años cumplidos Excepciones
AM
1. Ciclomotores de dos o tres ruedas y cuatriciclos ligeros.
Podrá estar limitado a la conducción de ciclomotores de tres ruedas y cuatriciclos ligeros.
2. Vehículos para personas de movilidad reducida.
15
A1
1. Motocicletas con una cilindrada máxima de 125 cm3, una potencia máxima de 11 kW y una relación potenc

## 1.5 Embeddings e indexación en ChromaDB

- **Modelo**: `models/gemini-embedding-001`. Se ha comprobado con `client.models.list()` qué modelos de embeddings ofrece la API (`gemini-embedding-001`, `gemini-embedding-2` y `gemini-embedding-2-preview`). Se elige `gemini-embedding-001` porque es el modelo estable de texto y admite el parámetro `task_type`: LangChain usa `RETRIEVAL_DOCUMENT` al indexar y `RETRIEVAL_QUERY` al consultar, lo que mejora la recuperación de pregunta → pasaje.
- **Dimensión 768** (`output_dimensionality`): el modelo genera 3072 dimensiones, pero está entrenado con *Matryoshka Representation Learning*, así que se puede truncar a 768 perdiendo muy poca calidad. A cambio, `chroma_db/` ocupa cuatro veces menos, lo que importa porque se versiona en git para desplegar en Streamlit Cloud.
- **Distancia coseno** en la colección (`hnsw:space = cosine`).
- **ChromaDB persistente** en `chroma_db/`, colección `normativa_trafico`.
- **Capa gratuita de la API**: se indexa en **lotes de 20 chunks** con **15 s de pausa** entre lotes y, si la API devuelve 429/`RESOURCE_EXHAUSTED`, **reintentos con espera exponencial** (30 s, 60 s, 120 s…).
- **Idempotente, reanudable e incremental**: el ID de cada chunk incluye un hash de su contenido. Si la colección ya está completa, no se reindexa. Si se cortó a medias (p. ej. por la cuota diaria), al relanzar solo se indexan los chunks que faltan. Si cambia la limpieza o el chunking, los chunks obsoletos se borran y solo se indexan los nuevos. Para rehacer el índice desde cero: `kb.build_knowledge_base(force=True)`.

> ⏱️ La primera ejecución tarda unos 20 minutos. Las siguientes son instantáneas.

In [7]:
vectorstore = kb.build_knowledge_base(force=False)
print("Documentos en la colección:", vectorstore._collection.count())

Chunks totales: 1321 | ya indexados: 0 | pendientes: 1321
  Lote 1/67 indexado (20 chunks)
  Lote 2/67 indexado (20 chunks)
  Lote 3/67 indexado (20 chunks)
  Lote 4/67 indexado (20 chunks)
  Lote 5/67 indexado (20 chunks)
  Lote 6/67 indexado (20 chunks)
  Lote 7/67 indexado (20 chunks)
  Lote 8/67 indexado (20 chunks)
  Lote 9/67 indexado (20 chunks)
  Lote 10/67 indexado (20 chunks)
  Lote 11/67 indexado (20 chunks)
  Lote 12/67 indexado (20 chunks)
  Lote 13/67 indexado (20 chunks)
  Lote 14/67 indexado (20 chunks)
  Lote 15/67 indexado (20 chunks)
  Lote 16/67 indexado (20 chunks)
  Lote 17/67 indexado (20 chunks)
  Lote 18/67 indexado (20 chunks)
  Lote 19/67 indexado (20 chunks)
  Lote 20/67 indexado (20 chunks)
  Lote 21/67 indexado (20 chunks)
  Lote 22/67 indexado (20 chunks)
  Lote 23/67 indexado (20 chunks)
  Lote 24/67 indexado (20 chunks)
  Lote 25/67 indexado (20 chunks)
  Lote 26/67 indexado (20 chunks)
  Lote 27/67 indexado (20 chunks)
  Lote 28/67 indexado (20 chunks)

## 1.6 Verificación del retrieval (sin LLM)

Antes de añadir el LLM se comprueba que el retriever devuelve los fragmentos correctos para preguntas típicas sobre motos.

### Retrieval híbrido (BM25 + vectores)

`get_retriever(k=5)` combina dos búsquedas y fusiona sus rankings con **Reciprocal Rank Fusion** (RRF):

- **Vectorial** (embeddings de Gemini en Chroma): capta el significado aunque cambien las palabras («¿es obligatorio el casco?» ~ «deberán utilizar cascos de protección homologados»).
- **BM25** (palabras clave, `rank_bm25`): acierta con **códigos cortos y términos exactos** que los embeddings distinguen mal: `A2`, `B`, `125 cm3`, `anexo II`… El texto se pasa a minúsculas, se le quitan las tildes y las stopwords. El índice BM25 se construye con los chunks guardados en Chroma, así que no hace falta tener los PDF (importante en Streamlit Cloud).
- **RRF**: para cada chunk se calcula `score = Σ peso / (60 + rango)` sobre las dos listas (top-20 de cada una) y se queda el top-5. Un chunk que sale arriba en ambas listas gana. Se usan pesos iguales.

Se puede elegir con `get_retriever(k, mode="hybrid" | "vector" | "bm25")`.

Para cada consulta se muestran los 5 chunks recuperados con su norma, artículo y página, y la **distancia coseno** del chunk según la búsqueda vectorial (menor = más parecido; `—` si el chunk solo viene de BM25).

Lo que se espera encontrar:

| Consulta | Dónde debería estar la respuesta |
|---|---|
| Edad mínima para el A2 | Reglamento de Conductores, art. 4 (fila A2 de la tabla: 18 años) |
| Motos que se pueden llevar con el B | Reglamento de Conductores, art. 4 (B con más de 3 años de antigüedad → motos del A1: 125 cm3 / 11 kW) |
| Casco obligatorio | Reglamento de Circulación, art. 118 (cascos y otros elementos de protección) |
| Pérdida de puntos | Ley de Tráfico: anexo II (infracciones que restan puntos) y art. 64 |
| Velocidad máxima de motos en autovía | Reglamento de Circulación, art. 48 (autopista y autovía: 120 km/h para turismos y motocicletas) |

In [8]:
retriever = kb.get_retriever(k=5)                       # híbrido (por defecto)
retriever_vec = kb.get_retriever(k=5, mode="vector")    # solo vectorial, para comparar
vectorstore = kb.get_vectorstore()

CONSULTAS = [
    "¿Cuál es la edad mínima para obtener el permiso de conducción A2?",
    "¿Qué motos puedo conducir con el permiso de la clase B?",
    "¿Es obligatorio el uso del casco en motocicleta?",
    "¿Qué infracciones suponen la pérdida de puntos del permiso de conducción?",
    "¿Cuál es la velocidad máxima de las motocicletas en autovía?",
]

# Comprobación automática: ¿está en el top-5 el chunk que contiene la respuesta?
ESPERADOS = [
    lambda d: "clase A2: 18 años" in d.page_content,
    lambda d: "clase B, con una antigüedad superior a tres años, las motocicletas" in d.page_content,
    lambda d: d.metadata["norma"].endswith("Circulación") and "Artículo 118" in d.metadata["articulos"],
    lambda d: d.metadata["norma"].startswith("Ley") and "Anexo II" in d.metadata["articulos"],
    lambda d: "Autopista y autovía 120" in d.page_content,
]

def mostrar_resultados(i, max_chars=500):
    consulta = CONSULTAS[i]
    docs = retriever.invoke(consulta)
    dist = {d.metadata["chunk_id"]: s for d, s in vectorstore.similarity_search_with_score(consulta, k=20)}
    print("=" * 110)
    print(f"CONSULTA: {consulta}")
    print("=" * 110)
    for r, d in enumerate(docs, 1):
        m = d.metadata
        s = dist.get(m["chunk_id"])
        ok = "  ✅ RESPUESTA ESPERADA" if ESPERADOS[i](d) else ""
        print(f"[{r}] {m['norma']} | {m['articulos']} | pág. {m['pagina']} | "
              f"distancia {f'{s:.3f}' if s is not None else '—'}{ok}")
        texto = d.page_content.split("\n", 1)[-1].replace("\n", " ")   # sin la cabecera de contexto
        print("    " + (texto[:max_chars] + ("…" if len(texto) > max_chars else "")))
        print()

In [9]:
mostrar_resultados(0)

CONSULTA: ¿Cuál es la edad mínima para obtener el permiso de conducción A2?
[1] Reglamento General de Conductores | — | pág. 7 | distancia 0.240
    Destaca la implantación del acceso progresivo como opción para obtener los permisos de conducción de determinados tipos de vehículos, como por ejemplo el de la nueva clase de permiso A2 que autoriza a conducir motocicletas de potencia media. Igualmente, se prevé la posibilidad de autorizar con el permiso de clase B la conducción de conjuntos de vehículos que excedan de 3500 kg, sin rebasar los 4250 kg, tras la superación de de una prueba de control de aptitudes y comportamientos que podrá ser su…

[2] Reglamento General de Conductores | Artículo 4 | pág. 11 | distancia 0.265
    Artículo 4. Clases de permiso de conducción y edad requerida para obtenerlo. 1. Todas las clases de permiso de conducción de las que sea titular una persona deberán constar en un único documento con expresión de las categorías de vehículos cuya conducción autorizan

In [10]:
mostrar_resultados(1)

CONSULTA: ¿Qué motos puedo conducir con el permiso de la clase B?
[1] Reglamento General de Conductores | Artículo 4 | pág. 12 | distancia 0.228
    Se podrán conducir dentro del territorio nacional con el permiso de la clase B, con una antigüedad superior a tres años, las motocicletas cuya conducción autoriza el permiso de la clase A1. En el caso de que el permiso de la clase B se encuentre sometido a adaptaciones, restricciones u otras limitaciones en personas, vehículos o de circulación, se estará a lo dispuesto en el artículo 5.7, segundo párrafo. B + E Conjuntos de vehículos acoplados compuestos por un vehículo tractor de los que autori…

[2] Reglamento General de Conductores | Artículo 5 | pág. 14 | distancia 0.234
    6. (Suprimido) 7. El permiso de las clases B, B+E, C1, C1+E, C, C+E, D1, D1+E, D y D+E no autoriza a conducir motocicletas con o sin sidecar. En el supuesto de que el permiso de la clase B en vigor, con una antigüedad superior a tres años, esté sometido a adaptacio

In [11]:
mostrar_resultados(2)

CONSULTA: ¿Es obligatorio el uso del casco en motocicleta?
[1] Reglamento General de Circulación | Artículo 118 | pág. 53 | distancia 0.288
    Artículo 118. Cascos y otros elementos de protección. 1. Los conductores y pasajeros de motocicletas o motocicletas con sidecar, de vehículos de tres ruedas y cuadriciclos, de ciclomotores y de vehículos especiales tipo «quad», deberán utilizar los siguientes elementos de protección: a) Cascos de protección homologados conforme al Reglamento UNECE R22, adecuadamente abrochados, cuando circulen en todo tipo de vías. b) Guantes de protección cuando circulen por vías interurbanas. La persona titula…

[2] Ley de Tráfico y Seguridad Vial | Artículo 47 | pág. 28 | distancia 0.302
    Artículo 47. Cinturón, casco y restantes elementos de seguridad. El conductor y ocupantes de vehículos a motor y ciclomotores están obligados a utilizar el cinturón de seguridad, el casco y demás elementos de protección en los términos que reglamentariamente se determine

In [12]:
mostrar_resultados(3)

CONSULTA: ¿Qué infracciones suponen la pérdida de puntos del permiso de conducción?
[1] Ley de Tráfico y Seguridad Vial | Anexo II | pág. 63 | distancia 0.224
    ANEXO II Infracciones que llevan aparejada la pérdida de puntos El titular de un permiso o licencia de conducción que sea sancionado en firme en vía administrativa por la comisión de alguna de las infracciones que a continuación se relacionan, perderá el número de puntos que, para cada una de ellas, se señala a continuación: Puntos 1. Conducir con una tasa de alcohol superior a la reglamentariamente establecida: Valores mg/l aire espirado, más de 0,50 (profesionales y titulares de permisos de co…

[2] Ley de Tráfico y Seguridad Vial | Artículo 64 | pág. 33 | distancia 0.232
    Artículo 64. Pérdida de puntos. 1. El número de puntos inicialmente asignado al titular de un permiso o licencia de conducción se verá reducido por cada sanción firme en vía administrativa que se le imponga por la comisión de infracciones graves o muy 

In [13]:
mostrar_resultados(4)

CONSULTA: ¿Cuál es la velocidad máxima de las motocicletas en autovía?
[1] Reglamento General de Circulación | Artículo 48 | pág. 27 | distancia 0.260
    Artículo 48. Velocidades máximas en vías fuera de poblado. 1. Las velocidades máximas que no deberán ser rebasadas, salvo en los supuestos previstos en el artículo 51, son las siguientes: a) Turismos, motocicletas, autocaravanas de masa máxima autorizada igual o inferior a 3.500 kg, Pick-up Camiones, tractocamiones, furgonetas, autocaravanas de masa máxima autorizada superior a 3.500 kg, vehículos articulados, automóviles con remolque y resto de vehículos Autobuses, vehículos derivados de turism…

[2] Reglamento General de Circulación | Artículo 36 | pág. 22 | distancia 0.288
    3. Se podrá habilitar el uso extraordinario del arcén de la derecha por motocicletas en aquellos tramos previamente acordados entre el titular de la vía y el organismo autónomo Jefatura Central de Tráfico o, en su caso, la autoridad autonómica o local respon

In [14]:
# Comparación: ¿en qué posición aparece la respuesta esperada? (vectorial vs. híbrido)
def posicion(ret, i):
    for r, d in enumerate(ret.invoke(CONSULTAS[i]), 1):
        if ESPERADOS[i](d):
            return r
    return None

comparacion = pd.DataFrame([{
    "Consulta": c,
    "Solo vectorial": posicion(retriever_vec, i),
    "Híbrido (BM25 + vectores)": posicion(retriever, i),
} for i, c in enumerate(CONSULTAS)]).set_index("Consulta")
comparacion = comparacion.map(lambda p: f"✅ top-{int(p)}" if pd.notna(p) else "❌ no está")
comparacion

,top1,top2,top3,top4,top5
¿Cuál es la edad mínima para obtener el permiso de conducció,Conductores · — (p.7),Conductores · Artículo 4 (p.11),Conductores · Artículo 74 (p.38),Conductores · Artículo 6 (p.15),Conductores · Artículo 74 (p.38)
¿Qué motos puedo conducir con el permiso de la clase B?,Conductores · Artículo 4 (p.12),Conductores · Artículo 5 (p.14),Conductores · Anexo VII (p.103),Conductores · Artículo 5 (p.14),Conductores · — (p.7)
¿Es obligatorio el uso del casco en motocicleta?,Circulación · Artículo 118 (p.53),Vial · Artículo 47 (p.28),Circulación · Artículo 116 (p.52),Circulación · Artículo 118 (p.53),Circulación · Artículo 118 (p.53)
¿Qué infracciones suponen la pérdida de puntos del permiso d,Vial · Anexo II (p.63),Vial · Artículo 64 (p.33),Vial · Artículo 64 (p.33),Vial · Artículo 65 (p.33),Vial · Anexo II (p.63)
¿Cuál es la velocidad máxima de las motocicletas en autovía?,Circulación · Artículo 48 (p.27),Circulación · Artículo 36 (p.22),Circulación · Artículo 48 (p.27),Vial · Artículo 20 (p.21),Circulación · Artículo 41 (p.25)


### Análisis de los resultados

**Primera versión** (solo vectorial, sin normalizar la tabla ni cabecera de contexto):

| Consulta | Resultado |
|---|---|
| Edad mínima A2 | ❌ **Fallo.** Se recuperaba el art. 4, pero el chunk que terminaba justo en «A2», sin la edad. El chunk con «A2 … 35 kW … 18» no aparecía en el top-5 porque empezaba a mitad de la tabla («2. Vehículos para personas de movilidad reducida. 16 A2…») y su embedding no tenía nada de «edad mínima» ni de «permiso». En el top-1 salía el preámbulo (que menciona el A2 pero no la edad). |
| Motos con el B | ✅ top-1: art. 4 (B + 3 años → motos del A1) y top-2: art. 5.7. |
| Casco | ✅ top-1: art. 118 del Reglamento de Circulación, más el art. 47 de la Ley y el art. 116. |
| Pérdida de puntos | ✅ top-1: anexo II de la Ley (tabla de puntos) y art. 64. |
| Velocidad en autovía | ✅ top-1: art. 48. La tabla también sale aplanada (`Autopista y autovía 120 90 100`), pero el valor está en el chunk y la cabecera de columnas («Turismos, motocicletas…») va primero, así que el LLM puede interpretarla. |

**Cambios aplicados**: (1) normalización de la tabla de clases de permiso, (2) cabecera de contexto `[norma · artículo]` en cada chunk, (3) retrieval híbrido BM25 + vectores con RRF. Al probar BM25 solo con los chunks nuevos, el chunk del A2 sale el **1.º**.

**Segunda versión**: *(completar con la tabla de comparación de arriba)*